In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"

# Folders for each method
imputed_dirs = {
    "SoftImpute": "imputed_softimpute",
    "MAGIC": "imputed_h5ad",
    "KNN": "imputed_knn",
    "Mean": "imputed_mean",
    "GAN": "imputed_gan",
    "Iterative": "imputed_iterative"
}

# -----------------------------
# Helper: Ensure dense float32 arrays
# -----------------------------
def to_dense_array(X):
    """Convert sparse or matrix-like to a dense float32 numpy array."""
    if hasattr(X, "toarray"):  # for csr_matrix, csc_matrix
        X = X.toarray()
    elif hasattr(X, "A"):      # for np.matrix
        X = X.A
    return np.array(X, dtype=np.float32)

# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt.var_names_make_unique()
adata_gt.obs_names_make_unique()
adata_gt.X = np.nan_to_num(to_dense_array(adata_gt.X),
                           nan=0.0, posinf=0.0, neginf=0.0)

# -----------------------------
# Helper: Gene-wise correlation
# -----------------------------
def compute_gene_corr(X_gt, X_imp, gene_names):
    corrs = []
    for i, g in enumerate(gene_names):
        x = X_gt[:, i]
        y = X_imp[:, i]

        # Skip constant genes
        if np.std(x) < 1e-8 or np.std(y) < 1e-8:
            continue

        try:
            r, _ = pearsonr(x, y)
        except Exception:
            r, _ = spearmanr(x, y)

        if np.isfinite(r):
            corrs.append(r)

    if len(corrs) == 0:
        return np.nan
    return np.nanmean(corrs)

# -----------------------------
# Evaluate all methods
# -----------------------------
results = []

for method, imputed_dir in imputed_dirs.items():
    if not os.path.exists(imputed_dir):
        print(f"⚠️ Skipping {method}, folder not found: {imputed_dir}")
        continue

    for fname in os.listdir(imputed_dir):
        if not fname.endswith(".h5ad"):
            continue

        fpath = os.path.join(imputed_dir, fname)
        try:
            adata_imp = sc.read_h5ad(fpath)
            adata_imp.var_names_make_unique()
            adata_imp.obs_names_make_unique()
            adata_imp.X = np.nan_to_num(to_dense_array(adata_imp.X),
                                        nan=0.0, posinf=0.0, neginf=0.0)

            # Align genes properly
            common_genes = adata_gt.var_names.intersection(adata_imp.var_names)

            if len(common_genes) == 0:
                print(f"⚠️ No common genes for {method} - {fname}. Skipping.")
                continue

            # Force same order of genes
            adata_gt_sub = adata_gt[:, common_genes].copy()
            adata_imp_sub = adata_imp[:, common_genes].copy()

            X_gt_sub = to_dense_array(adata_gt_sub.X)
            X_imp_sub = to_dense_array(adata_imp_sub.X)

            # Debugging: check shape match
            if X_gt_sub.shape != X_imp_sub.shape:
                print(f"⚠️ Shape mismatch for {method} - {fname}: {X_gt_sub.shape} vs {X_imp_sub.shape}")
                continue

            # Compute correlation
            mean_corr = compute_gene_corr(X_gt_sub, X_imp_sub, adata_gt_sub.var_names)

            # Parse metadata (mfXX, runX)
            mf, run = None, None
            for part in fname.split("_"):
                if part.startswith("mf"):
                    mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
                if part.startswith("run"):
                    run = int(part.replace("run", "").replace(".h5ad", ""))

            results.append({
                "file": fname,
                "method": method,
                "MeanGeneCorr": mean_corr,
                "missing_fraction": mf,
                "run": run
            })

            print(f"{method} - {fname}: Mean corr = {mean_corr:.3f} "
                  f"(n_genes={len(common_genes)})")

        except Exception as e:
            print(f"❌ Error with {method} - {fname}: {e}")

# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No correlations computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["MeanGeneCorr", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(10, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="MeanGeneCorr",
    hue="method",
    errorbar="sd",
    capsize=0.1,
    palette="Set2"
)
plt.title("Gene-wise Correlation across Missing Fractions (All Methods)")
plt.ylabel("Mean Gene-wise Correlation")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()
